In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.stats import wasserstein_distance
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ')

In [ ]:
MAMMO_TRAIN = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/tabular_ft_train (1).csv'
MAMMO_VAL = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/tabular_ft_val.csv'
MAMMO_TEST = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/tabular_final_test.csv'

US_TRAIN = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/train_predictions (1).csv'
US_VAL = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/validation_predictions (1).csv'
US_TEST = '/kaggle/input/datasets/habibashhefny/ps-mammo-us/test_predictions (1).csv'

OUTPUT_DIR = '/kaggle/working'

mammo = {
 'ft_train': pd.read_csv(MAMMO_TRAIN),
 'ft_val': pd.read_csv(MAMMO_VAL),
 'final_test': pd.read_csv(MAMMO_TEST),
}
us = {
 'ft_train': pd.read_csv(US_TRAIN),
 'ft_val': pd.read_csv(US_VAL),
 'final_test': pd.read_csv(US_TEST),
}

for split in mammo:
 mammo[split] = mammo[split].rename(columns={
 'full_img_path': 'mammo_path',
 'PS_tabular': 'PS_mammo'
 })

for split in us:
 us[split] = us[split].rename(columns={
 'image_path': 'us_path',
 'ps_score': 'PS_us'
 })

# ── Summary ───────────────────────────────────────────────────────────────────
print('=== Mammogram ===' )
for split, df in mammo.items():
 print(f' {split}: {len(df)} rows | label dist: {df["label"].value_counts().to_dict()}')
print()
print('=== Ultrasound ===')
for split, df in us.items():
 print(f' {split}: {len(df)} rows | label dist: {df["label"].value_counts().to_dict()}')

## 2. Optimal Transport Pairing Function

In [ ]:
def ot_pair_split(mammo_df: pd.DataFrame,
 us_df: pd.DataFrame,
 split_name: str,
 max_ps_diff: float = 0.15) -> pd.DataFrame:
 all_pairs = []

 for label in [0, 1]:
 label_name = 'Benign' if label == 0 else 'Malignant'

 m = mammo_df[mammo_df['label'] == label].reset_index(drop=True)
 u = us_df[us_df['label'] == label].reset_index(drop=True)

 if len(m) == 0 or len(u) == 0:
 print(f' [{split_name}] {label_name}: SKIP — empty group')
 continue

 # ── Cost matrix ────────────────────────────────────────────────────────
 # shape: (n_mammo, n_us)
 cost = np.abs(
 m['PS_mammo'].values[:, np.newaxis] -
 u['PS_us'].values[np.newaxis, :]
 )

 wd_before = wasserstein_distance(m['PS_mammo'].values, u['PS_us'].values)

 # ── Hungarian Algorithm ────────────────────────────────────────────────
 row_idx, col_idx = linear_sum_assignment(cost)

 # ── Build pairs DataFrame ──────────────────────────────────────────────
 paired = pd.DataFrame({
 'mammo_path': m.loc[row_idx, 'mammo_path'].values,
 'us_path': u.loc[col_idx, 'us_path'].values,
 'PS_mammo': m.loc[row_idx, 'PS_mammo'].values,
 'PS_us': u.loc[col_idx, 'PS_us'].values,
 'PS_diff': cost[row_idx, col_idx],
 'label': label,
 'split': split_name,
 })

 wd_after = paired['PS_diff'].mean()
 improvement = (wd_before - wd_after) / wd_before * 100 if wd_before > 0 else 0

 print(f' [{split_name}] {label_name}: '
 f'mammo={len(m)}, us={len(u)} → {len(paired)} pairs '
 f'| mean PS_diff={wd_after:.4f} '
 f'| improvement={improvement:.1f}%')

 before_filter = len(paired)
 paired = paired[paired['PS_diff'] <= max_ps_diff].reset_index(drop=True)
 removed = before_filter - len(paired)
 if removed > 0:
 print(f' Filtered {removed} pairs with PS_diff > {max_ps_diff}')

 all_pairs.append(paired)

 if not all_pairs:
 return pd.DataFrame()

 return pd.concat(all_pairs, ignore_index=True)

print('Function defined ')

## 3. run pairing on 3 splits

In [ ]:
print('Running OT Pairing on all 3 splits...\n')

paired_splits = {}

for split_name in ['ft_train', 'ft_val', 'final_test']:
 print(f'--- {split_name} ---')
 paired = ot_pair_split(
 mammo_df = mammo[split_name],
 us_df = us[split_name],
 split_name = split_name,
 max_ps_diff = 0.15
 )
 paired_splits[split_name] = paired
 print(f' Total pairs: {len(paired)}\n')

# ── Concat all splits ─────────────────────────────────────────────────────────
all_pairs = pd.concat(list(paired_splits.values()), ignore_index=True)

print('=' * 55)
print('SUMMARY')
print('=' * 55)
for split_name, df in paired_splits.items():
 print(f' {split_name}: {len(df)} pairs | label dist: {df["label"].value_counts().to_dict()}')
print(f' TOTAL: {len(all_pairs)} pairs')

## 4. Analysis of Pairing

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('OT Pairing Quality — All Splits', fontsize=14, fontweight='bold')

for row_i, split_name in enumerate(['ft_train', 'ft_val', 'final_test']):
 df = paired_splits[split_name]
 if len(df) == 0:
 continue

 # ── PS_diff distribution ──────────────────────────────────────────────────
 ax = axes[row_i][0]
 for label, color, name in [(0, '#42A5F5', 'Benign'), (1, '#EF5350', 'Malignant')]:
 s = df[df['label'] == label]['PS_diff']
 if len(s) > 0:
 ax.hist(s, bins=20, alpha=0.65, color=color,
 label=f'{name} (mean={s.mean():.3f})', density=True)
 ax.set_title(f'{split_name} — PS_diff distribution')
 ax.set_xlabel('|PS_mammo - PS_us|'); ax.set_ylabel('Density')
 ax.legend(fontsize=8); ax.grid(alpha=0.3)

 # ── Scatter: PS_mammo vs PS_us ────────────────────────────────────────────
 ax = axes[row_i][1]
 for label, color, name in [(0, '#42A5F5', 'Benign'), (1, '#EF5350', 'Malignant')]:
 s = df[df['label'] == label]
 ax.scatter(s['PS_mammo'], s['PS_us'],
 alpha=0.4, s=10, color=color, label=name)
 ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect match')
 ax.set_xlabel('PS_mammo'); ax.set_ylabel('PS_us')
 ax.set_title(f'{split_name} — PS_mammo vs PS_us')
 ax.legend(fontsize=8); ax.grid(alpha=0.3)

 # ── PS_mammo & PS_us distribution per label ───────────────────────────────
 ax = axes[row_i][2]
 for label, color, name in [(0, '#42A5F5', 'Benign'), (1, '#EF5350', 'Malignant')]:
 s = df[df['label'] == label]
 ax.hist(s['PS_mammo'], bins=15, alpha=0.5, color=color,
 label=f'{name} mammo', density=True, linestyle='solid')
 ax.hist(s['PS_us'], bins=15, alpha=0.3, color=color,
 label=f'{name} us', density=True, linestyle='dashed',
 histtype='step', linewidth=2)
 ax.set_xlabel('PS'); ax.set_ylabel('Density')
 ax.set_title(f'{split_name} — PS distributions')
 ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ot_pairing_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ')

## 5. Save Pairs

In [ ]:
import os

for split_name, df in paired_splits.items():
 out_path = os.path.join(OUTPUT_DIR, f'pairs_{split_name}.csv')
 df.to_csv(out_path, index=False)
 print(f'Saved → {out_path} ({len(df)} pairs)')

all_out = os.path.join(OUTPUT_DIR, 'all_pairs.csv')
all_pairs.to_csv(all_out, index=False)
print(f'Saved → {all_out} ({len(all_pairs)} total pairs)')

print()
print('Columns in each file:')
for col in all_pairs.columns:
 print(f' {col}')
print()
print('Final summary:')
print(all_pairs.groupby(['split', 'label']).size().rename('count').to_string())